# Examples of dynamical systems

In [ ]:
#    APM41012EP course notebook - Chapter 7 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Examples of dynamical systems
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Brusselator model

The dynamics of the oscillating reaction discovered by Belousov and Zhabotinsky can be modelled by the so-called Brusselator model, depending on two parameters $a$ and $b$:

$$
\left\{\begin{aligned}
{\mathrm d_t} y_1 & = a - (b+1) y_1 + y_1^2y_2\\
{\mathrm d_t} y_2 & = b y_1 - y_1^2y_2
\end{aligned}\right.
$$

In [ ]:
class brusselator_model:

    def __init__(self, a, b) :
        self.a = a
        self.b = b
    
    def fcn(self, t, y):
        y1, y2 = y
        a = self.a
        b = self.b 
        y1_dot = a - (b+1)*y1 + y1*y1*y2 
        y2_dot = b*y1 - y1*y1*y2  
        return np.array([y1_dot, y2_dot])

### Numerical solution

In [ ]:
tini = 0. 
tend = 40.
yini = (1.5, 3.)

bm = brusselator_model(a=0.7, b=3)
fcn = bm.fcn  
sol_exa = solve_ivp(fcn, (tini, tend), yini, method="RK45", rtol=1.e-10, atol=1.e-10)

fig = make_subplots(rows=2, cols=1, subplot_titles=("Phase space", "Solutions"), vertical_spacing=0.15)
fig.add_trace(go.Scatter(x=sol_exa.y[0], y=sol_exa.y[1], mode="lines", showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=sol_exa.t, y=sol_exa.y[0], mode="lines", name="y1"), row=2, col=1)
fig.add_trace(go.Scatter(x=sol_exa.t, y=sol_exa.y[1], mode="lines", name="y2"), row=2, col=1)

steps = []
for i, a_i in enumerate(np.arange(0.7, 1.71, 0.1)):
    bm = brusselator_model(a=a_i, b=3)
    fcn = bm.fcn  
    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="RK45", rtol=1.e-10, atol=1.e-10)
    step = dict(method="update", label = f"{a_i:.2f}", args=[{"x":[sol_exa.y[0], sol_exa.t, sol_exa.t],
                                                              "y":[sol_exa.y[1], sol_exa.y[0], sol_exa.y[1]]}])
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'a = '}, steps=steps)]

fig.update_xaxes(title_text="u1", range=[0, 6], row=1)
fig.update_yaxes(title_text="u2", row=1)
fig.update_xaxes(title_text="t", row=2)
fig.update_layout(sliders=sliders, height=800, legend=dict(x=0.01, y=0.37, bgcolor='rgba(0,0,0,0.1)'))
fig['layout']['sliders'][0]['pad']=dict(t= 50)
fig.show()

In order to illustrate the concepts presented in the lecture, we use $b=3.0$, $y_1^0=1.5$ and $y_2^0=3$ and let $a$ vary between $0.7$ and $1.7$. For this set of parameters, the dynamics admits a limit cycle for large times and, after a short transient period, the behaviour of the system becomes almost periodic, as can be observed when $a$ is small enough (critical point $y_1=a$, $y_2=b/a$, Hopf bifurcation at $a=\sqrt{2}$), or spirals towards the stable critical point when $a$ is large enough.

## Lorenz system


The attractor and the associated equations were published in 1963 by Edward Lorenz. 
Mathematically, the coupling of the Earth's atmosphere with the ocean is described by the coupled Navier-Stokes system of partial differential equations of fluid mechanics. This system of equations was far too complicated to solve numerically on the computers available at the time of Lorenz, who introduced a very simplified model of these equations in order to study a particular physical situation: the Rayleigh-Bénard convection phenomenon.

The Lorenz system is a system of three ordinary differential equations: 
$$
\left\{\begin{aligned}
{\mathrm d_t} x & = \sigma \, (y - x)\\
{\mathrm d_t} y & = \rho x - y - xz\\ 
{\mathrm d_t} z & = xy - \beta z
\end{aligned}\right.
$$

where 
$\sigma$, $\rho$ and $\beta$
 are three strictly positive, fixed parameters. 
 Since the system comes from the simplification of the equations governing Rayleigh-Bénard convection, the parameters are named after their physical origin: $\sigma$ is the Prandtl number, and $\rho$, loosely called the "Rayleigh number", is in fact the ratio of the Rayleigh number Ra to the critical Rayleigh number.

The dynamical variables $x$, $y$ and $z$ represent the state of the system at each instant. Their physical interpretation is the following: $x$ is proportional to the intensity of the convective motion, $y$ is proportional to the temperature difference between the ascending and descending currents, and $z$ is proportional to the deviation of the vertical temperature profile from a linear profile.

In [ ]:
class lorenz_model():

    def __init__(self, sigma, rho, beta):
        self.sigma = sigma
        self.rho = rho
        self.beta = beta

    def fcn(self, t, xyz):
        x, y, z = xyz
        sigma = self.sigma
        rho = self.rho
        beta = self.beta
        x_dot = sigma*(y-x)
        y_dot = rho*x - y - x*z
        z_dot = x*y - beta*z
        return (x_dot, y_dot, z_dot)

### Numerical solution

In [ ]:
sigma=10; rho=28; beta=8/3
sol_ini = (-10, -7, 35) 
tini = 0.; tend = 100.

lm = lorenz_model(sigma=sigma, rho=rho, beta=beta)
fcn = lm.fcn  
    
sol = solve_ivp(fcn, (tini, tend), sol_ini, method="RK45", rtol=1.e-8, atol=1.e-10)
x = sol.y[0]; y = sol.y[1]; z = sol.y[2]

fig = make_subplots(rows=2, cols=2, specs=[[{}, {}],[{"colspan": 2}, None]], vertical_spacing=0.1,
                    subplot_titles=("Phase space (xy)", "Phase space (xz)", "Solution x"))
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", showlegend=False, marker_color='rgb(76,114,176)' ), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=z, mode="lines", showlegend=False, marker_color='rgb(76,114,176)'), row=1, col=2)
fig.add_trace(go.Scatter(x=sol.t, y=x, mode="lines", name="x", showlegend=False, marker_color='rgb(76,114,176)'), row=2, col=1)

fig.update_layout(height=1000, title="Numerical solution")
fig.show()

One often sets $\sigma=10$ and $\beta= 8/3$, $\rho$ remaining variable. The system exhibits a chaotic behaviour for $\rho = 28$, after some time in a pseudo-periodic regime that depends on $\rho$. The threshold for the onset of the chaotic regime lies between $24.29$ and $24.30$ for these values of 
$\sigma$ and $\beta$.

### Perturbation of the initial conditions

We perturb the first component of the initial conditions by adding $10^{-8}$.

In [ ]:
sol_ini_pert = (-10.000000001, -7, 35)

sol_pert = solve_ivp(fcn, (tini, tend), sol_ini_pert, method="RK45", rtol=1.e-8, atol=1.e-10)
x_pert = sol_pert.y[0]; y_pert = sol_pert.y[1]; z_pert = sol_pert.y[2]

fig = make_subplots(rows=2, cols=2, specs=[[{}, {}],[{"colspan": 2}, None]], vertical_spacing=0.1,
                    subplot_titles=("Phase space (xy)", "Phase space (xz)", "Solution x"))
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", marker_color='rgb(76,114,176)', legendgroup="1", name="sol."), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=z, mode="lines", showlegend=False, marker_color='rgb(76,114,176)', legendgroup="1"), row=1, col=2)
fig.add_trace(go.Scatter(x=sol.t, y=x, mode="lines", name="x", showlegend=False, marker_color='rgb(76,114,176)', legendgroup="1"), row=2, col=1)
fig.add_trace(go.Scatter(x=x_pert, y=y_pert, mode="lines", marker_color='rgb(221,132,82)', legendgroup="2", name="pert. sol."), row=1, col=1)
fig.add_trace(go.Scatter(x=x_pert, y=z_pert, mode="lines", showlegend=False, marker_color='rgb(221,132,82)', legendgroup="2"), row=1, col=2)
fig.add_trace(go.Scatter(x=sol.t, y=x_pert, mode="lines", name="x", showlegend=False, marker_color='rgb(221,132,82)', legendgroup="2"), row=2, col=1)

fig.update_layout(height=1000, legend=dict(orientation="h", y=1.1), margin=dict(t=180))
fig.show()

## Celestial mechanics: the three-body problem

Let us consider the Sun-Jupiter-Saturn system where, for the sake of simplicity, we neglect the other bodies and influences of the solar system. In 1687, Isaac Newton, inspired by the three laws of Kepler, proposed the universal law of gravitation, according to which all cosmic objects attract each other with equal (but opposite) forces,
proportional to the product of their masses and inversely proportional 
to the square of the distance separating them. It is this law that we are going to use to compute
the position of the planets. The gravitational force $\vec F_{S\rightarrow P}$ applied by a body $S$ to a body $P$ is given by the following formula:

$$
\vec F_{S\rightarrow P} = -\vec F_{P\rightarrow S} = - \frac{G\,m_S\,m_P}{d^2} \vec{u}, 
$$

where $G$ is the universal gravitational constant, $m_S$ and $m_P$ are the masses of the different
bodies $S$ and $P$, $d$ is the (Euclidean) distance between $S$ and $P$, and $\vec{u}$ is a unit vector oriented from $S$ to $P$.


We represent the positions of these planets
by three functions of time, $q_i(t)\in\mathbb{R}^3$, $i\in{0,1,2}$
where the index $i=0$ corresponds
 to the Sun, $i=1$ corresponds to Jupiter, and $i=2$ corresponds to Saturn. 
 The respective masses of the three bodies are denoted by $m_i$, $i\in{0,1,2}$. 
We also consider the momenta $p_i(t)\in\mathbb{R}^3$, $i\in{0,1,2}$.
Newton's second law of motion is then expressed as follows: 

$$
{{\mathrm d}}_t p_0 = \vec F_{Sa\rightarrow S} + \vec F_{J\rightarrow S},\quad
{{\mathrm d}}_t p_1 = \vec F_{S\rightarrow J} + \vec F_{Sa\rightarrow J} \quad
{{\mathrm d}}_t p_2 = \vec F_{S\rightarrow Sa} + \vec F_{J\rightarrow Sa},
$$

We therefore end up with a system of 6 equations (or, more precisely, 18, since each $p_i$ and $q_i$ belongs to $\mathbb{R}^3$): 

$$
\left\{\begin{aligned}
q_i^′(t) & = \frac{1}{m_i} p_i\\
p_i^′(t) & = -G \sum_{j \neq i} \frac{m_i m_j}{\lVert q_i - q_j \rVert^2} \frac{q_i - q_j}{\lVert q_i - q_j \rVert}
\end{aligned}\right.
\quad \quad i \in 0, 1, 2.
$$


In [ ]:
class three_body_model:

    def __init__(self):
        self.m0 = 1.00000597682
        self.m1 = 9.54786104043e-4
        self.m2 = 2.85583733151e-4
        self.G = 2.95912208286e-4

    def fcn(self, t, y):
        q = y[0:9]
        p = y[9:18]
        
        q_dot = np.zeros(9)
        p_dot = np.zeros(9)
        
        m0 = self.m0
        m1 = self.m1
        m2 = self.m2
        G  = self.G
        
        q_dot[0:3] = p[0:3]/m0
        q_dot[3:6] = p[3:6]/m1
        q_dot[6:9] = p[6:9]/m2
        
        p_dot[0:3] = -( G*m0*m1*((q[0:3]-q[3:6])/np.power(np.linalg.norm(q[0:3]-q[3:6]),3)) 
                       +G*m0*m2*((q[0:3]-q[6:9])/np.power(np.linalg.norm(q[0:3]-q[6:9]),3)))
        p_dot[3:6] = -( G*m1*m0*((q[3:6]-q[0:3])/np.power(np.linalg.norm(q[3:6]-q[0:3]),3))
                       +G*m1*m2*((q[3:6]-q[6:9])/np.power(np.linalg.norm(q[3:6]-q[6:9]),3)))
        p_dot[6:9] = -( G*m2*m0*((q[6:9]-q[0:3])/np.power(np.linalg.norm(q[6:9]-q[0:3]),3)) 
                       +G*m2*m1*((q[6:9]-q[3:6])/np.power(np.linalg.norm(q[6:9]-q[3:6]),3)))
        
        return np.concatenate((q_dot, p_dot))
    
    def hamiltonian(self, y):
        m0 = self.m0
        m1 = self.m1
        m2 = self.m2
        G = self.G
        nt = y.shape[1]
        neq = y.shape[0]
        q = y[0:neq//2]
        p = y[neq//2:neq]
        ham = np.zeros(nt)

        for i in range(nt):
            ham[i] = ( 0.5*( (1/m0)*np.dot(p[0:3,i],p[0:3,i]) 
                            +(1/m1)*np.dot(p[3:6,i],p[3:6,i]) 
                            +(1/m2)*np.dot(p[6:9,i],p[6:9,i]) ) 
                      - G *( (m0*m1)/np.linalg.norm(q[0:3,i]-q[3:6,i]) 
                            +(m0*m2)/np.linalg.norm(q[0:3,i]-q[6:9,i])
                            +(m1*m2)/np.linalg.norm(q[3:6,i]-q[6:9,i]) ) )

        return ham

The data of the simulation are reproduced in the table below: 


|  body  |  mass (relative to the Sun)  | position (A.U.) |  velocity (A.U. / day)  |
|---    |:-:    |:-:    |:-:    |
| Sun  |  $ m_0 = 1.00000597682$ |  $\left(0, 0, 0\right)^t$ |  $\left(0,0,0\right)^t$ |
| Jupiter |  $m_1 = 9.54786104043 \, 10^{-4}$ | $\left(-3.5023653,-3.8169847,-1.5507963\right)^t$  | $\left(0.00565429,-0.00412490, -0.00190589\right)^t$ |
| Saturn |  $m_2 = 2.85583733151 \, 10^{-4}$ | $\left(9.0755314,-3.0458353,-1.6483708\right)^t$  | $\left(0.00168318, 0.00483525, 0.00192462\right)^t$   

where $m_0$ corresponds to the mass of the Sun together with the other planets of the solar system that are not taken into account here.


The initial positions and velocities of the Sun, Jupiter and Saturn at a given date 
(here 5 September 1994) are expressed in astronomical units, based on the Earth-Sun distance
(1 A.U. is about 150 million kilometres), and time is in Earth days.
These trajectories are almost in a plane, but they evolve in 3D. The Sun itself also moves slightly, but we represent the trajectories relative to the Sun, chosen as the reference and located at the origin. 

The Hamiltonian of the system is a first integral of the motion; it is the gravitational energy of the motion (sum of the kinetic energy and of the potential energy), which is conserved: 

$$
H = \frac{1}{2}\left(\frac{\lVert p_0 \rVert^2}{m_0} + \frac{\lVert p_1 \rVert^2}{m_1} + \frac{\lVert p_2 \rVert^2}{m_2} \right)               - G \left( \frac{m_0 m_1}{\lVert q_0 - q_1 \rVert} + \frac{m_0 m_2}{\lVert q_0 - q_2 \rVert} + \frac{m_1 m_2}{\lVert q_1 - q_2 \rVert} \right)
$$

In [ ]:
# initialization
m1 = 9.54786104043e-4
m2 = 2.85583733151e-4

qini = np.zeros(9)
qini[0] =  0.;        qini[1] =  0.;        qini[2] =  0.
qini[3] = -3.5023653; qini[4] = -3.8169847; qini[5] = -1.5507963
qini[6] =  9.0755314; qini[7] = -3.0458353; qini[8] = -1.6483708

pini = np.zeros(9)
pini[0] =  0.;            pini[1] =  0.;            pini[2] =  0.
pini[3] =  0.00565429*m1; pini[4] = -0.00412490*m1; pini[5] = -0.00190589*m1
pini[6] =  0.00168318*m2; pini[7] =  0.00483525*m2; pini[8] =  0.00192462*m2

yini = np.concatenate((qini, pini))

tini = 0.
tend = 11000.
nt = 1501

tbm = three_body_model()
fcn = tbm.fcn

sol = solve_ivp(fcn, (tini, tend), yini, method="RK45", rtol=1.e-12, atol=1.e-12)

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0.], y=[0.], mode="markers", marker=dict(size=12, color="orange"), name="Sun"))
fig.add_trace(go.Scatter(x=[qini[3]], y=[qini[4]], mode="markers", marker=dict(size=12, color="olive"), showlegend=False))
fig.add_trace(go.Scatter(x=sol.y[3]-sol.y[0], y=sol.y[4]-sol.y[1], marker_color="olive", name="Jupiter"))
fig.add_trace(go.Scatter(x=[qini[6]], y=[qini[7]], mode="markers", marker=dict(size=12, color="steelblue"), showlegend=False))
fig.add_trace(go.Scatter(x=sol.y[6]-sol.y[0], y=sol.y[7]-sol.y[1], marker_color="steelblue ", name="Saturn"))
fig.update_layout(height=500, xaxis_range=[-12, 12])
fig.show()

Simulation of the dynamics of the Sun-Saturn-Jupiter planetary system with a symplectic integrator (optimised composition method of order 8 with 15 stages, based on Störmer-Verlet). The inner trajectory is Jupiter's and the outer one is Saturn's. We have also plotted the evolution of the Hamiltonian of the system below, and the symplectic integrator makes it possible to guarantee that the numerical solution preserves the invariant over time.

### Hamiltonian

In [ ]:
ham = tbm.hamiltonian(sol.y)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sol.t, y=ham))
fig.update_layout(title="Hamiltonian")
fig.update_yaxes(range=[-3.15636e-8, -3.15634e-8], exponentformat='e')
fig.show()

## Population dynamics

### Lotka-Volterra model

In 1925 A.J. Lotka published a book entitled `Elements of Physical Biology`, in which he set out to represent the kinetics of populations living in a community by systems of differential equations.
In one of the chapters, he considers the example of a population of herbivorous animals feeding on plants. By analogy with the equations used for chemical kinetics, representing by $u_1(t)$ the total mass of plants and by $u_2(t)$ the total mass of herbivores at time $t$, Lotka proposes the following model:

$$
\left\{\begin{aligned}
{\mathrm d}_t u_1 & = u_1 \, (1-u_2)\\
{\mathrm d}_t u_2 & = u_2 \, (-k+u_1).
\end{aligned}\right.
$$

In [ ]:
class lotka_model:

    def __init__(self, alpha):
        self.alpha = alpha

    def fcn(self, t, u):
        u1, u2 = u
        alpha = self.alpha
        u1_dot = u1 * (1 - u2)
        u2_dot = u2 * (-alpha + u1)
        return (u1_dot, u2_dot)

In [ ]:
alpha = np.arange(0.5,2.6,0.2)

lm = lotka_model(alpha[0])
fcn = lm.fcn

uini = (0.5, 0.5)
tini = 0
tend = 200

tol = 1.e-8

sol = solve_ivp(fcn, (tini, tend), uini, rtol=tol, atol=tol)
u1 = sol.y[0]; u2 = sol.y[1]
inv = u1 + u2 - np.log(u2 * np.power(u1,alpha[0]))

fig = make_subplots(rows=2, cols=1, subplot_titles=("Phase space", "Solutions"), vertical_spacing=0.15)
fig.add_trace(go.Scatter(x=u1, y=u2, showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=u1, name="u1"), row=2, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=u2, name="u2"), row=2, col=1)

#create slider
steps = []
for alpha_i in alpha:
    lm = lotka_model(alpha_i)
    fcn = lm.fcn
    sol = solve_ivp(fcn, (tini, tend), uini, rtol=tol, atol=tol)
    u1 = sol.y[0]; u2 = sol.y[1]
    inv = u1 + u2 - np.log(u2 * np.power(u1,alpha_i))
    step = dict(method="update", label = f"{alpha_i:.2f}", args=[{"x": [u1,sol.t,sol.t], "y": [u2, u1, u2]}])
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'alpha = '}, steps=steps)]

fig.update_xaxes(title_text="u1", range=[0, 8], row=1)
fig.update_yaxes(title_text="u2", range=[0, 5], row=1)
fig.update_xaxes(title_text="t", row=2)
fig.update_yaxes(title_text="t", range=[0, 8], row=2)
fig.update_layout(sliders=sliders, height=800, legend=dict(x=0.01, y=0.37, bgcolor='rgba(0,0,0,0.1)'))
fig['layout']['sliders'][0]['pad']=dict(t=50)

fig.show()

### Rosenzweig-MacArthur model

Soon after the introduction of the Lotka-Volterra model, various modifications were proposed for the predation term. Indeed, in this model the number of prey killed by the predators is proportional to the product of the number of individuals of each population, i.e. proportional to $u_1(t)u_2(t)$. In other words, the number of prey killed per predator grows proportionally to the number of prey itself, and there is therefore no saturation or satiety effect. This is qualitatively what is observed for some bacterial populations. However, for more complex living organisms, such as animals, one expects a really different behaviour, where the predators cannot consume more than a maximal amount of prey per unit of time.

In 1959 the American ecologist Buzz Holling proposed three main types of modelling for the number of prey killed per
predator: the first one is that of the Lotka-Volterra model (type I) and the other two introduce a saturation effect when the number of prey exceeds a certain threshold (types II and III). Types II and III differ when the number of prey is very small and
make it possible to distinguish *generalist* from *specialist* predators.

It was in 1963 that the American ecologists Robert MacArthur and Michael L. Rosenzweig studied the following prey-predator model:

$$
\left\{\begin{aligned}
{\mathrm d}_t u_1  & = u_1 \, \Big( 1-\frac{u_1}{\gamma} \Big) - \frac{u_1 u_2}{1+u_1}\\
{\mathrm d}_t u_2  & = \beta u_2 \Big( \frac{u_1}{1+u_1} - \alpha \Big).  
\end{aligned}\right.
$$

The predation term is of type II.

In [ ]:
class rosenzweig_model:

    def __init__(self, alpha, K):
        self.alpha = alpha
        self.K = K

    def fcn(self, t, u):
        u1, u2 = u
        alpha = self.alpha
        K = self.K
        u1_dot = u1 * (1 - (u1/K)) - ((u1*u2)/(1+u1))
        u2_dot = u2 * ((u1/(1+u1)) - alpha)
        return (u1_dot, u2_dot)

In [ ]:
alpha = np.arange(0.2,0.65,0.05)
K = 2
rm = rosenzweig_model(alpha[0], K)
fcn = rm.fcn

uini = (1., 1.)
tini = 0
tend = 200

tol = 1.e-8

sol = solve_ivp(fcn, (tini, tend), uini, rtol=tol, atol=tol)
u1 = sol.y[0]; u2 = sol.y[1]

fig = make_subplots(rows=2, cols=1, subplot_titles=("Phase space", "Solutions"), vertical_spacing=0.15)
fig.add_trace(go.Scatter(x=u1, y=u2, showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=u1, name="u1", mode="lines"), row=2, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=u2, name="u2", mode="lines"), row=2, col=1)

#create slider
steps = []
for alpha_i in alpha:
    rm = rosenzweig_model(alpha_i, K)
    fcn = rm.fcn
    sol = solve_ivp(fcn, (tini, tend), uini, rtol=tol, atol=tol)
    u1 = sol.y[0]; u2 = sol.y[1]
    step = dict(method="update", label = f"{alpha_i:.2f}", args=[{"x": [u1,sol.t,sol.t], "y": [u2, u1, u2]}])
    steps.append(step)
sliders = [dict(currentvalue={'prefix': 'alpha = '}, steps=steps)]

fig.update_xaxes(title_text="u1", range=[-0.1, 2.5], row=1)
fig.update_yaxes(title_text="u2", range=[-0.1, 2.5], row=1)
fig.update_xaxes(title_text="t", row=2)
fig.update_yaxes(range=[-0.1, 2.5], row=2)
fig.update_layout(sliders=sliders, height=800, legend=dict(x=0.88, y=0.37, bgcolor='rgba(0,0,0,0.1)'))
fig['layout']['sliders'][0]['pad']=dict(t=50)

fig.show()